# Train-Ready Grammar Data EDA

This notebook focuses on the final cleaned dataset used for BabyLLaMA fine-tuning.

It is designed for files such as `../../data/processed/train_ready_grammar_data.jsonl`, and it uses the same official `count_word` method from `Language_acquisition.ipynb` to count words in `good` and `bad`.

In [55]:
import json
from pathlib import Path

import nltk
import pandas as pd
from nltk.tokenize import word_tokenize

In [56]:
for resource in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource)

In [57]:
TRAIN_READY_PATH = Path("../../data\\processed\\trial_generated_grammar_data_by_llm_prompt\\openai_prompt_3.jsonl")

In [58]:
def count_word(text):
    tokens = word_tokenize(text)
    return len(tokens)

In [59]:
def load_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

In [60]:
records = load_jsonl(TRAIN_READY_PATH)
df = pd.DataFrame(records)
print(f"Loaded {len(df)} train-ready records from {TRAIN_READY_PATH}")
df.head()

Loaded 1848 train-ready records from ..\..\data\processed\trial_generated_grammar_data_by_llm_prompt\openai_prompt_3.jsonl


,id,phenomenon,topic,good,bad,edit_type,prompt_family,parent_llm
0,openai_prompt_3_4d286caf4b,anaphor_agreement,family_home,The brothers proud of their work congratulated...,The brothers proud of their work congratulated...,reflexive_number_match,prompt_3,openai
1,openai_prompt_3_bf7153eb09,anaphor_agreement,family_home,Aunt Clara reminded herself to lock the back d...,Aunt Clara reminded themselves to lock the bac...,reflexive_number_match,prompt_3,openai
2,openai_prompt_3_b9be469d00,anaphor_agreement,school_classroom,The students taught themselves the new formula...,The students taught himself the new formula be...,reflexive_number_match,prompt_3,openai
3,openai_prompt_3_55adc0a6c7,anaphor_agreement,school_classroom,Nina allowed herself a short break after pract...,Nina allowed themselves a short break after pr...,reflexive_number_match,prompt_3,openai
4,openai_prompt_3_3dea095593,anaphor_agreement,household_objects,The heater switched itself off at midnight.,The heater switched himself off at midnight.,reflexive_animacy_match,prompt_3,openai


## Required Columns

In [61]:
required_columns = [
    "id",
    "phenomenon",
    "topic",
    "good",
    "bad",
    "edit_type",
    "prompt_family",
    "parent_llm",
]

missing_columns = [col for col in required_columns if col not in df.columns]
missing_columns

[]

## Official Word Counts for `good` and `bad`

In [62]:
df["good_word_count"] = df["good"].fillna("").map(count_word)
df["bad_word_count"] = df["bad"].fillna("").map(count_word)
df["pair_word_count"] = df["good_word_count"] + df["bad_word_count"]
df[["good", "bad", "good_word_count", "bad_word_count", "pair_word_count"]].head()

,good,bad,good_word_count,bad_word_count,pair_word_count
0,The brothers proud of their work congratulated...,The brothers proud of their work congratulated...,11,11,22
1,Aunt Clara reminded herself to lock the back d...,Aunt Clara reminded themselves to lock the bac...,10,10,20
2,The students taught themselves the new formula...,The students taught himself the new formula be...,11,11,22
3,Nina allowed herself a short break after pract...,Nina allowed themselves a short break after pr...,9,9,18
4,The heater switched itself off at midnight.,The heater switched himself off at midnight.,8,8,16


In [63]:
word_count_summary = {
    "num_records": int(len(df)),
    "total_good_words": int(df["good_word_count"].sum()),
    "total_bad_words": int(df["bad_word_count"].sum()),
    "total_pair_words": int(df["pair_word_count"].sum()),
    "avg_good_words": float(df["good_word_count"].mean()) if len(df) else 0.0,
    "avg_bad_words": float(df["bad_word_count"].mean()) if len(df) else 0.0,
    "avg_pair_words": float(df["pair_word_count"].mean()) if len(df) else 0.0,
    "min_good_words": int(df["good_word_count"].min()) if len(df) else 0,
    "max_good_words": int(df["good_word_count"].max()) if len(df) else 0,
    "min_bad_words": int(df["bad_word_count"].min()) if len(df) else 0,
    "max_bad_words": int(df["bad_word_count"].max()) if len(df) else 0,
}
word_count_summary

{'num_records': 1848,
 'total_good_words': 16804,
 'total_bad_words': 16434,
 'total_pair_words': 33238,
 'avg_good_words': 9.093073593073592,
 'avg_bad_words': 8.892857142857142,
 'avg_pair_words': 17.985930735930737,
 'min_good_words': 6,
 'max_good_words': 15,
 'min_bad_words': 3,
 'max_bad_words': 18}

## Dataset Overview

In [64]:
overview = {
    "num_records": int(len(df)),
    "num_unique_ids": int(df["id"].nunique(dropna=True)),
    "num_unique_good": int(df["good"].nunique(dropna=True)),
    "num_unique_bad": int(df["bad"].nunique(dropna=True)),
    "num_unique_phenomena": int(df["phenomenon"].nunique(dropna=True)),
    "num_unique_topics": int(df["topic"].nunique(dropna=True)),
    "num_unique_edit_types": int(df["edit_type"].nunique(dropna=True)),
    "num_unique_prompt_families": int(df["prompt_family"].nunique(dropna=True)),
    "num_unique_parent_llms": int(df["parent_llm"].nunique(dropna=True)),
}
overview

{'num_records': 1848,
 'num_unique_ids': 923,
 'num_unique_good': 923,
 'num_unique_bad': 922,
 'num_unique_phenomena': 12,
 'num_unique_topics': 10,
 'num_unique_edit_types': 30,
 'num_unique_prompt_families': 1,
 'num_unique_parent_llms': 1}

In [65]:
df.isna().sum().sort_values(ascending=False)

id                 0
phenomenon         0
topic              0
good               0
bad                0
edit_type          0
prompt_family      0
parent_llm         0
good_word_count    0
bad_word_count     0
pair_word_count    0
dtype: int64

## Counts by Phenomenon

In [66]:
df["phenomenon"].value_counts().rename_axis("phenomenon").reset_index(name="count")

,phenomenon,count
0,anaphor_agreement,168
1,argument_structure,168
2,binding,168
3,control_raising,168
4,determiner_noun_agreement,168
5,ellipsis,168
6,irregular_forms,168
7,npi_licensing,168
8,subject_verb_agreement,168
9,quantifiers,168


## Counts by Topic

In [67]:
df["topic"].value_counts().rename_axis("topic").reset_index(name="count")

,topic,count
0,family_home,302
1,school_classroom,272
2,shopping_market,208
3,household_objects,196
4,books_stories,184
5,chores_routines,176
6,park_outdoors,138
7,travel_transport,136
8,food_meals,120
9,animals_pets,116


## Counts by Parent LLM

In [68]:
df["parent_llm"].value_counts().rename_axis("parent_llm").reset_index(name="count")

,parent_llm,count
0,openai,1848


## Counts by Prompt Family

In [69]:
df["prompt_family"].value_counts().rename_axis("prompt_family").reset_index(name="count")

,prompt_family,count
0,prompt_3,1848


## Counts by Edit Type

In [70]:
df["edit_type"].value_counts().rename_axis("edit_type").reset_index(name="count")

,edit_type,count
0,determiner_number_match,168
1,number_agreement,168
2,reflexive_number_match,94
3,existential_there_quantifier,92
4,one_anaphora_modifier_position,86
5,resumptive_pronoun,84
6,ones_anaphora_modifier_position,82
7,existential_there_with_strong_quantifier,76
8,missing_object,72
9,unlicensed_reflexive,66


## Cross Tabs

In [71]:
pd.crosstab(df["phenomenon"], df["parent_llm"])

parent_llm,openai
phenomenon,
anaphor_agreement,168
argument_structure,168
binding,168
control_raising,168
determiner_noun_agreement,168
ellipsis,168
filler_gap,84
irregular_forms,168
island_effects,84


In [72]:
pd.crosstab(df["phenomenon"], df["prompt_family"])

prompt_family,prompt_3
phenomenon,
anaphor_agreement,168
argument_structure,168
binding,168
control_raising,168
determiner_noun_agreement,168
ellipsis,168
filler_gap,84
irregular_forms,168
island_effects,84


In [73]:
pd.crosstab(df["topic"], df["parent_llm"])

parent_llm,openai
topic,
animals_pets,116
books_stories,184
chores_routines,176
family_home,302
food_meals,120
household_objects,196
park_outdoors,138
school_classroom,272
shopping_market,208


## Length Analysis

In [74]:
df[["good_word_count", "bad_word_count", "pair_word_count"]].describe()

,good_word_count,bad_word_count,pair_word_count
count,1848.000000,1848.000000,1848.000000
mean,9.093074,8.892857,17.985931
std,1.638068,2.039245,3.607250
min,6.000000,3.000000,9.000000
25%,8.000000,8.000000,16.000000
50%,9.000000,9.000000,18.000000
75%,10.000000,10.000000,20.000000
max,15.000000,18.000000,33.000000


In [75]:
df["word_count_gap"] = (df["good_word_count"] - df["bad_word_count"]).abs()
df["word_count_gap"].describe()

count    1848.000000
mean        0.377706
std         0.754112
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         3.000000
Name: word_count_gap, dtype: float64

In [76]:
df.sort_values("pair_word_count", ascending=False)[
    ["id", "phenomenon", "topic", "good", "bad", "good_word_count", "bad_word_count", "pair_word_count"]
].head(20)

,id,phenomenon,topic,good,bad,good_word_count,bad_word_count,pair_word_count
1327,openai_prompt_3_01c590a286,island_effects,school_classroom,Which student did the teacher admire after rea...,Which student did the teacher admire after rea...,15,18,33
1285,openai_prompt_3_01c590a286,island_effects,school_classroom,Which student did the teacher admire after rea...,Which student did the teacher admire after rea...,15,18,33
1292,openai_prompt_3_106fde5e81,island_effects,books_stories,Which author did the fan point to in the lette...,Which author did the fan point to in the lette...,15,17,32
1291,openai_prompt_3_6304d72a1e,island_effects,school_classroom,What workbook did the tutor talk about in the ...,What workbook did the tutor talk about in the ...,15,17,32
1333,openai_prompt_3_6304d72a1e,island_effects,school_classroom,What workbook did the tutor talk about in the ...,What workbook did the tutor talk about in the ...,15,17,32
1334,openai_prompt_3_106fde5e81,island_effects,books_stories,Which author did the fan point to in the lette...,Which author did the fan point to in the lette...,15,17,32
1330,openai_prompt_3_7f509809b9,island_effects,school_classroom,Whose project did the counselor discuss after ...,Whose project did the counselor discuss after ...,14,17,31
1328,openai_prompt_3_3a6c190e27,island_effects,books_stories,What chapter did the critic praise in the nove...,What chapter did the critic praise in the nove...,14,17,31
1287,openai_prompt_3_aa8bc0fa89,island_effects,travel_transport,Which station did the planner mention during t...,Which station did the planner mention during t...,14,17,31
1288,openai_prompt_3_7f509809b9,island_effects,school_classroom,Whose project did the counselor discuss after ...,Whose project did the counselor discuss after ...,14,17,31


## Quality Checks on Final Training Data

In [77]:
quality_checks = {
    "good_equals_bad": int((df["good"] == df["bad"]).sum()),
    "duplicate_ids": int(df.duplicated(subset=["id"]).sum()),
    "duplicate_training_rows": int(df.duplicated(subset=["phenomenon", "topic", "good", "bad", "edit_type"]).sum()),
    "empty_good": int((df["good"].fillna("").str.strip() == "").sum()),
    "empty_bad": int((df["bad"].fillna("").str.strip() == "").sum()),
}
quality_checks

{'good_equals_bad': 6,
 'duplicate_ids': 925,
 'duplicate_training_rows': 925,
 'empty_good': 0,
 'empty_bad': 0}

In [78]:
df[df.duplicated(subset=["phenomenon", "topic", "good", "bad", "edit_type"], keep=False)].sort_values(
    ["phenomenon", "topic", "edit_type"]
)[["id", "phenomenon", "topic", "good", "bad", "edit_type", "prompt_family", "parent_llm"]].head(30)

,id,phenomenon,topic,good,bad,edit_type,prompt_family,parent_llm
6,openai_prompt_3_6bfea5238e,anaphor_agreement,family_home,My mother asked herself whether the plan was s...,My mother asked himself whether the plan was s...,reflexive_gender_match,prompt_3,openai
7,openai_prompt_3_10c2f6a0e1,anaphor_agreement,family_home,My uncle blamed himself for the late arrival.,My uncle blamed herself for the late arrival.,reflexive_gender_match,prompt_3,openai
13,openai_prompt_3_08f0e37195,anaphor_agreement,family_home,Dad reminded himself to buy milk on the way back.,Dad reminded herself to buy milk on the way back.,reflexive_gender_match,prompt_3,openai
24,openai_prompt_3_877883f77f,anaphor_agreement,family_home,Our grandmother caught herself humming in the ...,Our grandmother caught himself humming in the ...,reflexive_gender_match,prompt_3,openai
25,openai_prompt_3_bc0cad6883,anaphor_agreement,family_home,The child comforted himself with a blanket.,The child comforted herself with a blanket.,reflexive_gender_match,prompt_3,openai
30,openai_prompt_3_9ab5244e60,anaphor_agreement,family_home,My sister blamed herself for forgetting the key.,My sister blamed himself for forgetting the key.,reflexive_gender_match,prompt_3,openai
37,openai_prompt_3_70024b5aa5,anaphor_agreement,family_home,The uncle shaved himself before the wedding.,The uncle shaved herself before the wedding.,reflexive_gender_match,prompt_3,openai
90,openai_prompt_3_6bfea5238e,anaphor_agreement,family_home,My mother asked herself whether the plan was s...,My mother asked himself whether the plan was s...,reflexive_gender_match,prompt_3,openai
91,openai_prompt_3_10c2f6a0e1,anaphor_agreement,family_home,My uncle blamed himself for the late arrival.,My uncle blamed herself for the late arrival.,reflexive_gender_match,prompt_3,openai
97,openai_prompt_3_08f0e37195,anaphor_agreement,family_home,Dad reminded himself to buy milk on the way back.,Dad reminded herself to buy milk on the way back.,reflexive_gender_match,prompt_3,openai


## Samples

In [79]:
df.sample(min(10, len(df)), random_state=42) if len(df) else df

,id,phenomenon,topic,good,bad,edit_type,prompt_family,parent_llm,good_word_count,bad_word_count,pair_word_count,word_count_gap
351,openai_prompt_3_988206ffa5,binding,books_stories,Sara thought that the author would dedicate th...,Sara thought that the author would dedicate th...,nonlocal_reflexive,prompt_3,openai,12,12,24,0
1288,openai_prompt_3_7f509809b9,island_effects,school_classroom,Whose project did the counselor discuss after ...,Whose project did the counselor discuss after ...,complex_np_island,prompt_3,openai,14,17,31,3
1516,openai_prompt_3_4d3d45b76f,quantifiers,animals_pets,There is a kitten under the porch.,There is each kitten under the porch.,existential_there_quantifier,prompt_3,openai,8,8,16,0
1263,openai_prompt_3_d0d4d3f2ae,island_effects,travel_transport,Which ticket did the agent stamp at the counter?,Which did the agent stamp ticket at the counter?,left_branch_extraction,prompt_3,openai,10,10,20,0
408,openai_prompt_3_649b351837,binding,books_stories,The siblings noticed that the ghost greeted th...,The siblings noticed that the ghost greeted th...,unlicensed_reflexive,prompt_3,openai,12,12,24,0
65,openai_prompt_3_114c4b0783,anaphor_agreement,household_objects,The clock set itself back an hour.,The clock set himself back an hour.,reflexive_animacy_match,prompt_3,openai,8,8,16,0
1192,openai_prompt_3_4375103dff,irregular_forms,family_home,We have driven to the store for milk.,We have drived to the store for milk.,irregular_auxiliary_form,prompt_3,openai,9,9,18,0
1640,openai_prompt_3_d22d17ec18,quantifiers,shopping_market,There was a jar beside the scales.,There was each jar beside the scales.,existential_there_quantifier,prompt_3,openai,8,8,16,0
1217,openai_prompt_3_81ebf4d47f,irregular_forms,park_outdoors,The apples were picked from the tree.,The apples were pickt from the tree.,irregular_participle_adjective,prompt_3,openai,8,8,16,0
1101,openai_prompt_3_26967e3aa9,irregular_forms,chores_routines,The shirts are washed after practice.,The shirts are wash after practice.,irregular_participle,prompt_3,openai,7,7,14,0


In [80]:
for phenomenon in sorted(df["phenomenon"].dropna().unique()[:5]):
    print(f"\n=== {phenomenon} ===")
    display(df[df["phenomenon"] == phenomenon][["id", "topic", "good", "bad", "edit_type"]].head(3))


=== anaphor_agreement ===


,id,topic,good,bad,edit_type
0,openai_prompt_3_4d286caf4b,family_home,The brothers proud of their work congratulated...,The brothers proud of their work congratulated...,reflexive_number_match
1,openai_prompt_3_bf7153eb09,family_home,Aunt Clara reminded herself to lock the back d...,Aunt Clara reminded themselves to lock the bac...,reflexive_number_match
2,openai_prompt_3_b9be469d00,school_classroom,The students taught themselves the new formula...,The students taught himself the new formula be...,reflexive_number_match



=== argument_structure ===


,id,topic,good,bad,edit_type
168,openai_prompt_3_706e2d0320,family_home,Maya scrubbed the stove after dinner.,Maya scrubbed after dinner.,missing_object
169,openai_prompt_3_9a25e8660b,family_home,The lamp stood beside the sofa.,The lamp stood the sofa.,illicit_object
170,openai_prompt_3_3aecca07b5,family_home,Evan hung the picture in the hallway.,Evan hung the picture.,missing_required_complement



=== binding ===


,id,topic,good,bad,edit_type
336,openai_prompt_3_5a8fa44d67,family_home,The twins praised themselves after the recital.,The twins praised them after the recital.,unlicensed_reflexive
337,openai_prompt_3_6e8cdbc6c7,family_home,Nadia reminded herself to lock the back door.,Nadia reminded herself to lock the back door.,reflexive_subject
338,openai_prompt_3_b2174030ff,school_classroom,The teacher told us that we should pack our no...,The teacher told us that ourselves should pack...,reflexive_subject



=== control_raising ===


,id,topic,good,bad,edit_type
504,openai_prompt_3_52f20841d9,travel_transport,There appears to be a delay at the platform.,There plans to be a delay at the platform.,expletive_there_with_raising
505,openai_prompt_3_21c259b119,school_classroom,There seems to be a quiz on Friday.,There wants to be a quiz on Friday.,expletive_there_with_raising
506,openai_prompt_3_4156647167,books_stories,There happens to be a map in the chapter.,There hopes to be a map in the chapter.,expletive_there_with_raising



=== determiner_noun_agreement ===


,id,topic,good,bad,edit_type
672,openai_prompt_3_8bbd0fc0b4,shopping_market,These apples look sweeter than the rest.,This apples look sweeter than the rest.,determiner_number_match
673,openai_prompt_3_7781b99cd1,shopping_market,That loaf was still warm at noon.,Those loaf was still warm at noon.,determiner_number_match
674,openai_prompt_3_13b35ca456,shopping_market,We chose those oranges from the back stall.,We chose that oranges from the back stall.,determiner_number_match


## Optional Export

In [81]:
EXPORT_CSV = False

if EXPORT_CSV:
    df.to_csv("train_ready_grammar_data_eda_export.csv", index=False)
    print("Saved train_ready_grammar_data_eda_export.csv")